# 考古模型项目启动

目标：构建一个专门辅助考古研究的模型系统。

路线：

```text
RAG 知识库 + 微调模型 + 证据规则 prompt + 聊天界面
```

注意：不要从零训练大模型。

## 1. 核心原则

- **知识放 RAG**：考古报告、论文、表格、图版说明。
- **方法放微调**：固定抽取、分类、写作流程。
- **规则放 prompt**：必须引用、不确定标“待核”、地层优先。
- **界面隐藏检索**：用户只聊天，后台自动查资料。

## 2. 推荐工具栈

| 用途 | 推荐工具 |
|------|---------|
| 最低门槛 RAG | RAGFlow |
| 代码自定义 RAG | LlamaIndex / LangChain |
| 向量库 | Chroma / Qdrant |
| PDF/OCR 解析 | MinerU / PyMuPDF / PaddleOCR |
| 微调 | LLaMA-Factory / Unsloth |
| 基座模型 | Qwen2.5-7B-Instruct / 14B |
| 本地部署 | Ollama / vLLM |
| 聊天界面 | Open WebUI / Dify |

## 3. 第一步：资料结构

```text
corpus/
  facts/                # 事实资料：报告、简报、表格、图版
    excavation_reports/
    tomb_tables/
    artifact_catalogs/
    captions/
  methods/              # 方法资料：类型学、地层学、断代
    typology/
    stratigraphy/
    dating/
  glossary/             # 术语表
    terms.md
    artifact_names.md
  workflows/            # 工作流模板
    tomb_analysis.md
    artifact_classification.md
  rules/                # 证据规则
    evidence_policy.md
    citation_policy.md
```

## 4. 元数据格式

每段资料都要带元数据：

```json
{
  "title": "XX遗址发掘报告",
  "source_type": "excavation_report",
  "site": "XX遗址",
  "region": "河南",
  "period": "战国",
  "page": 123,
  "feature_id": "M12",
  "topic": ["墓葬", "陶器", "分期"]
}
```

## 5. 最小可行路线

1. 选 50 篇考古 PDF
2. 用 MinerU 或 PyMuPDF 解析成文本
3. 按墓葬/器物/图版切片，附元数据
4. 用 LlamaIndex + Chroma 建 RAG
5. 测试 30 个考古问题，看检索准不准
6. 写 500 条高质量微调样本
7. 用 Qwen2.5-7B + QLoRA 微调
8. 把 RAG + 微调模型接到聊天界面

## 6. 安装依赖（示例）

先只装 RAG 相关的，微调后面再加。

In [ ]:
# 安装 RAG 基础依赖
# !uv add llama-index chromadb openai pypdf
print('依赖安装命令：uv add llama-index chromadb openai pypdf')

## 7. LlamaIndex RAG 最小示例

In [ ]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

# 读取文本资料
documents = SimpleDirectoryReader("./corpus").load_data()

# 建索引
index = VectorStoreIndex.from_documents(documents)

# 保存索引
index.storage_context.persist("./storage")

# 查询
query_engine = index.as_query_engine()
response = query_engine.query("M12 的墓葬形制和出土器物是什么？")
print(response)

## 8. 微调样本格式示例

```json
{
  "instruction": "从考古报告段落中抽取墓葬信息。",
  "input": "M12，竖穴土坑墓，墓向180°，出土陶鬲1、陶豆2、陶罐1。",
  "output": {
    "遗迹编号": "M12",
    "墓葬形制": "竖穴土坑墓",
    "墓向": "180°",
    "随葬品": ["陶鬲", "陶豆", "陶罐"],
    "年代判断": "资料不足，不能单独断代",
    "证据": "原文仅提供形制、墓向和随葬品，缺少地层关系、图版比较或测年数据。"
  }
}
```

## 9. 证据规则 prompt

```text
你是考古研究助手。

必须遵守：
1. 只基于检索到的资料回答。
2. 每个判断必须引用来源。
3. 不确定则明确标注“待核”。
4. 地层关系优先于器物风格。
5. 冲突资料必须并列说明。
6. 区分原文事实、研究推断和个人假设。
```

## 10. 下一步

先做 50 篇资料的 RAG 原型，测试 30 个问题。

RAG 可用之后，再进入微调阶段。